# Bu Dersi Google Colab'da Çalıştır

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BILSEM-BT/Python/blob/main/22-PythonRegresyon.ipynb)

Bu notebook GitHub üzerinde ders dokümanı olarak yayımlanır. Kodları çalıştırmak ve üzerinde denemeler yapmak için yukarıdaki **Open in Colab** butonunu kullanabilirsiniz.

### Nasıl çalışacağız?

1. **Open in Colab** butonuna tıklayın.
2. Açılan notebook'taki kod hücrelerini `▶` düğmesiyle çalıştırın.
3. Kodları değiştirerek farklı sonuçları deneyin.
4. Çalışmalarınız kendi Colab çalışma alanınızda tutulur; bu GitHub'daki ana ders dosyasını değiştirmez.

> **Önemli:** GitHub'daki bu dosya dersin ana ve değiştirilmeyen kaynağıdır. Colab'da yaptığınız değişiklikler bu dosyaya otomatik olarak yazılmaz.

---

# 22 - Python ile Regresyon ve Sayısal Tahmin Uygulamaları

**Niyazi Sayın BİLSEM**  
**Bilişim Teknolojileri Dersi**  
**Ders Öğretmeni: Ersin ŞANLI**

Bir önceki derste sınıflandırma problemi üzerinde çalıştık ve bir verinin hangi sınıfa ait olduğunu tahmin eden ilk makine öğrenmesi modelimizi oluşturduk.

Bu derste hedefimiz farklıdır:

**Bir kategori değil, sayısal bir değer tahmin edeceğiz.**

Örnek problemler:

- çalışma süresinden sınav puanı tahmini,
- ev özelliklerinden fiyat tahmini,
- geçmiş satışlardan gelecek satış miktarı tahmini,
- sensör değerlerinden enerji tüketimi tahmini,
- ürün özelliklerinden talep tahmini.

Bu tür problemlere **regresyon** problemleri denir.

Bu dersin sonunda öğrencinin:

- regresyon problemini tanıması,
- bağımsız ve hedef değişken kavramlarını kullanması,
- basit doğrusal regresyon oluşturması,
- çoklu doğrusal regresyon kullanması,
- eğitim ve test verisini ayırması,
- MAE, MSE, RMSE ve R² ölçülerini yorumlaması,
- gerçek ve tahmin değerlerini karşılaştırması,
- residual kavramını tanıması,
- farklı regresyon modellerini karşılaştırması,
- yeni veriler için sayısal tahmin üretmesi,
- eğitilmiş modeli dosyaya kaydedip tekrar yüklemesi

hedeflenmektedir.


# 1. Regresyon Nedir?

Makine öğrenmesinde hedef değişken sayısal bir değer olduğunda çoğunlukla bir **regresyon problemi** ile karşı karşıyayız.

Örnek:

```text
Girdi:
Ev alanı = 120 m²
Oda sayısı = 3
Bina yaşı = 8

Hedef:
Tahmini fiyat = 4.250.000
```

Modelin çıktısı bir sınıf etiketi değil, sayısal bir değerdir.


# 2. Sınıflandırma ve Regresyon Farkı

### Sınıflandırma

Çıktı bir kategori veya sınıftır.

```text
E-posta → Spam
Çiçek → Setosa
Fotoğraf → Kedi
```

### Regresyon

Çıktı sayısal değerdir.

```text
Ev → 4.250.000
Sıcaklık → 24.7
Satış → 1850
```

Bir makine öğrenmesi problemine başlarken ilk sorulardan biri şudur:

**Tahmin etmek istediğim hedef bir sınıf mı, sayısal değer mi?**


# 3. İlk Problem: Çalışma Süresi ve Sınav Puanı

Basit bir veri kümesi oluşturalım.

Her öğrencinin:

- haftalık çalışma süresi,
- sınav puanı

bulunsun.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

veri = pd.DataFrame({
    "CalismaSaati": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "Puan": [42, 48, 55, 61, 67, 72, 79, 84, 90, 95]
})

veri


# 4. Veriyi Görselleştirmek

In [ ]:
plt.scatter(
    veri["CalismaSaati"],
    veri["Puan"]
)

plt.xlabel("Haftalık Çalışma Saati")
plt.ylabel("Sınav Puanı")
plt.title("Çalışma Süresi ve Sınav Puanı")
plt.grid()
plt.show()


Grafikte çalışma süresi arttıkça sınav puanının da genel olarak arttığını görüyoruz.

Bu ilişkiyi bir doğrusal model ile yaklaşık olarak temsil etmeye çalışabiliriz.


# 5. Özellik ve Hedef

Makine öğrenmesindeki gösterimimizi hatırlayalım:

```text
X → modelin kullandığı özellikler
y → tahmin edilmek istenen hedef
```


In [ ]:
X = veri[["CalismaSaati"]]
y = veri["Puan"]

print(X)
print()
print(y)


Tek bir özellik kullansak bile scikit-learn çoğunlukla `X` verisini iki boyutlu tablo biçiminde bekler.

Bu nedenle:

```python
veri[["CalismaSaati"]]
```

kullandık.


# 6. Linear Regression

İlk regresyon algoritmamız `LinearRegression` olacaktır.


In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()

print(model)


# 7. Modeli Eğitmek

Sınıflandırma dersinde olduğu gibi regresyonda da model eğitiminde `fit()` kullanılır.


In [ ]:
model.fit(X, y)

print("Model eğitildi.")


# 8. Modelin Katsayısı

Doğrusal regresyon basit durumda şu yapıyı öğrenmeye çalışır:

```text
y = a × x + b
```

Burada:

- `a` → katsayı,
- `b` → sabit terimdir.


In [ ]:
print("Katsayı:", model.coef_[0])
print("Sabit terim:", model.intercept_)


Modelimizin yaklaşık denklemini yazabiliriz.


In [ ]:
a = model.coef_[0]
b = model.intercept_

print(
    f"Tahmini Puan = {a:.3f} × Çalışma Saati + {b:.3f}"
)


# 9. İlk Tahmin

Haftada 7.5 saat çalışan bir öğrenci için tahmin yapalım.


In [ ]:
yeni_veri = pd.DataFrame({
    "CalismaSaati": [7.5]
})

tahmin = model.predict(yeni_veri)

print("Tahmin edilen puan:", tahmin[0])


# 10. Birden Fazla Tahmin

Birden fazla yeni örnek aynı anda modele verilebilir.


In [ ]:
yeni_ogrenciler = pd.DataFrame({
    "CalismaSaati": [2.5, 5.5, 8.5]
})

tahminler = model.predict(
    yeni_ogrenciler
)

for saat, puan in zip(
    yeni_ogrenciler["CalismaSaati"],
    tahminler
):
    print(
        f"{saat} saat -> {puan:.2f}"
    )


# 11. Regresyon Doğrusunu Çizmek

Modelin tahmin ettiği doğruyu gerçek verilerin üzerine çizelim.


In [ ]:
x_cizgi = pd.DataFrame({
    "CalismaSaati": np.linspace(1, 10, 100)
})

y_cizgi = model.predict(x_cizgi)

plt.scatter(
    veri["CalismaSaati"],
    veri["Puan"],
    label="Gerçek Veriler"
)

plt.plot(
    x_cizgi["CalismaSaati"],
    y_cizgi,
    label="Regresyon Doğrusu"
)

plt.xlabel("Çalışma Saati")
plt.ylabel("Puan")
plt.title("Doğrusal Regresyon")
plt.legend()
plt.grid()
plt.show()


# 12. Gerçek Değer ve Tahmin

Modelin eğitim verileri için tahminlerini inceleyelim.


In [ ]:
veri["Tahmin"] = model.predict(X)

veri


# 13. Residual Nedir?

Gerçek değer ile tahmin arasındaki fark **residual / artık** olarak adlandırılır.

Basit olarak:

```text
Residual = Gerçek - Tahmin
```


In [ ]:
veri["Residual"] = (
    veri["Puan"] -
    veri["Tahmin"]
)

veri


# 14. Residual Değerlerini İncelemek

In [ ]:
plt.scatter(
    veri["Tahmin"],
    veri["Residual"]
)

plt.axhline(
    y=0
)

plt.xlabel("Tahmin Edilen Puan")
plt.ylabel("Residual")
plt.title("Residual Grafiği")
plt.grid()
plt.show()


İdeal bir modelde residual değerlerinin belirgin bir sistematik örüntü göstermemesi beklenir.

Residual analizi daha ileri regresyon çalışmalarında model hatalarını anlamak için kullanılır.


# 15. Neden Eğitim ve Test Verisi Ayırmalıyız?

Az önce bütün veriyi hem eğitim hem değerlendirme için kullandık.

Bu öğrenme amacıyla kolaydır ancak model başarısını güvenilir biçimde ölçmek için yeterli değildir.

Modelin daha önce görmediği veriler üzerinde test edilmesi gerekir.


# 16. Daha Büyük Bir Örnek Veri Kümesi

Şimdi 120 öğrencilik yapay ancak daha gerçekçi bir veri kümesi üretelim.


In [ ]:
rng = np.random.default_rng(42)

calisma_saati = rng.uniform(
    1,
    15,
    120
)

gurultu = rng.normal(
    0,
    6,
    120
)

puan = (
    35 +
    4.2 * calisma_saati +
    gurultu
)

puan = np.clip(
    puan,
    0,
    100
)

ogrenci_df = pd.DataFrame({
    "CalismaSaati": calisma_saati,
    "Puan": puan
})

ogrenci_df.head()


Burada gerçek dünya verilerindeki düzensizliği taklit etmek için rastgele gürültü ekledik.

Aynı çalışma süresine sahip öğrencilerin puanları gerçek hayatta tamamen aynı olmak zorunda değildir.


# 17. Veri Kümesinin Özeti

In [ ]:
ogrenci_df.describe()


# 18. Yeni Veriyi Görselleştirmek

In [ ]:
plt.scatter(
    ogrenci_df["CalismaSaati"],
    ogrenci_df["Puan"]
)

plt.xlabel("Çalışma Saati")
plt.ylabel("Puan")
plt.title("120 Öğrencilik Veri Kümesi")
plt.grid()
plt.show()


# 19. Train-Test Ayrımı

In [ ]:
from sklearn.model_selection import train_test_split

X = ogrenci_df[["CalismaSaati"]]
y = ogrenci_df["Puan"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Eğitim:", X_train.shape)
print("Test:", X_test.shape)


# 20. Modeli Sadece Eğitim Verisiyle Eğitmek

In [ ]:
regresyon = LinearRegression()

regresyon.fit(
    X_train,
    y_train
)

print("Model eğitildi.")


# 21. Test Verisi İçin Tahmin

In [ ]:
y_pred = regresyon.predict(
    X_test
)

print(y_pred[:10])


# 22. Tahminleri Karşılaştırmak

In [ ]:
sonuclar = pd.DataFrame({
    "Gercek": y_test.to_numpy(),
    "Tahmin": y_pred
})

sonuclar["Hata"] = (
    sonuclar["Gercek"] -
    sonuclar["Tahmin"]
)

sonuclar.head(10)


# 23. Regresyonda Model Değerlendirme

Sınıflandırmada accuracy kullanmıştık.

Regresyonda sayısal tahminlerin ne kadar hata yaptığını ölçmek için farklı metrikler kullanırız.

Bu derste:

- MAE
- MSE
- RMSE
- R²

öğreneceğiz.


In [ ]:
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)


# 24. MAE — Mean Absolute Error

MAE, tahminlerin gerçek değerlerden mutlak olarak ortalama ne kadar uzak olduğunu gösterir.

Düşük olması tercih edilir.


In [ ]:
mae = mean_absolute_error(
    y_test,
    y_pred
)

print("MAE:", mae)


Örneğin MAE yaklaşık 5 ise modelin tahminleri gerçek puanlardan ortalama olarak yaklaşık 5 puan uzaklıkta düşünülebilir.


# 25. MSE — Mean Squared Error

MSE hataların karesini alır.

Bu nedenle büyük hatalara daha fazla ağırlık verir.


In [ ]:
mse = mean_squared_error(
    y_test,
    y_pred
)

print("MSE:", mse)


MSE'nin birimi hedef değişkenin biriminin karesi olduğu için doğrudan yorumlanması MAE kadar kolay olmayabilir.


# 26. RMSE — Root Mean Squared Error

MSE'nin karekökünü alarak tekrar hedef değişkenin birimine dönebiliriz.


In [ ]:
rmse = np.sqrt(mse)

print("RMSE:", rmse)


RMSE de düşük olduğunda daha iyi hata performansına işaret eder.


# 27. R² Skoru

R², modelin hedef değişkendeki değişimi ne kadar açıklayabildiği hakkında bilgi verir.

Genel olarak:

- `1.0` → kusursuz uyum,
- `0.0` → yalnızca ortalama tahmini yapan basit bir modele benzer,
- negatif değer → ortalama tahmininden daha kötü performans

anlamına gelebilir.

R² farklı veri kümeleri arasında tek başına karşılaştırma ölçüsü olarak düşünülmemelidir.


In [ ]:
r2 = r2_score(
    y_test,
    y_pred
)

print("R²:", r2)


# 28. Bütün Metrikleri Birlikte Yazdırmak

In [ ]:
print(f"MAE  : {mae:.3f}")
print(f"MSE  : {mse:.3f}")
print(f"RMSE : {rmse:.3f}")
print(f"R²   : {r2:.3f}")


# 29. Gerçek ve Tahmin Değerleri Grafiği

In [ ]:
plt.scatter(
    y_test,
    y_pred
)

minimum = min(
    y_test.min(),
    y_pred.min()
)

maksimum = max(
    y_test.max(),
    y_pred.max()
)

plt.plot(
    [minimum, maksimum],
    [minimum, maksimum]
)

plt.xlabel("Gerçek Puan")
plt.ylabel("Tahmin Puan")
plt.title("Gerçek ve Tahmin Değerleri")
plt.grid()
plt.show()


Noktalar referans çizgisine ne kadar yakınsa tahminler gerçek değerlere o kadar yakındır.


# 30. Baseline Model Neden Önemlidir?

Bir yapay zeka modelinin başarısını yalnızca kendi skoruna bakarak değerlendirmek yeterli değildir.

Basit bir yöntemden gerçekten daha iyi olup olmadığını görmek gerekir.

Regresyonda temel bir baseline olarak sürekli eğitim hedefinin ortalamasını tahmin eden model kullanılabilir.


In [ ]:
from sklearn.dummy import DummyRegressor

baseline = DummyRegressor(
    strategy="mean"
)

baseline.fit(
    X_train,
    y_train
)

baseline_pred = baseline.predict(
    X_test
)

print(
    "Baseline R²:",
    r2_score(
        y_test,
        baseline_pred
    )
)


# 31. Kendi Modelimiz ile Baseline Karşılaştırması

In [ ]:
karsilastirma = pd.DataFrame({
    "Model": [
        "Ortalama Baseline",
        "Linear Regression"
    ],
    "R2": [
        r2_score(
            y_test,
            baseline_pred
        ),
        r2_score(
            y_test,
            y_pred
        )
    ],
    "MAE": [
        mean_absolute_error(
            y_test,
            baseline_pred
        ),
        mean_absolute_error(
            y_test,
            y_pred
        )
    ]
})

karsilastirma


Bir modelin baseline'dan daha iyi olması, öğrendiği özelliklerin tahmine gerçekten katkı sağladığını göstermeye yardımcı olur.


# 32. Çoklu Regresyon

Gerçek problemlerde hedef değer çoğunlukla tek bir özellikten etkilenmez.

Örneğin bir evin fiyatı:

- alan,
- oda sayısı,
- bina yaşı,
- merkeze uzaklık

gibi birçok özellikten etkilenebilir.

Birden fazla özelliğin kullanıldığı doğrusal modele **çoklu doğrusal regresyon** diyebiliriz.


# 33. Ev Fiyatı Veri Kümesi Oluşturalım

Dersin internet bağlantısına ihtiyaç duymadan çalışabilmesi için kontrollü yapay bir veri kümesi oluşturacağız.


In [ ]:
rng = np.random.default_rng(100)

n = 300

alan = rng.integers(
    60,
    251,
    n
)

oda = rng.integers(
    1,
    7,
    n
)

yas = rng.integers(
    0,
    41,
    n
)

merkeze_uzaklik = rng.uniform(
    0.5,
    30,
    n
)

gurultu = rng.normal(
    0,
    250000,
    n
)

fiyat = (
    500000
    + alan * 28000
    + oda * 120000
    - yas * 18000
    - merkeze_uzaklik * 35000
    + gurultu
)

ev_df = pd.DataFrame({
    "Alan": alan,
    "Oda": oda,
    "Yas": yas,
    "MerkezeUzaklik": merkeze_uzaklik,
    "Fiyat": fiyat
})

ev_df.head()


Bu veri gerçek emlak fiyatı değildir.

Makine öğrenmesi sürecini güvenli ve tekrar üretilebilir biçimde öğrenmek için oluşturulmuş eğitim verisidir.


# 34. Ev Verisinin Boyutu

In [ ]:
print(ev_df.shape)


# 35. Temel İstatistikler

In [ ]:
ev_df.describe()


# 36. Eksik Veri Kontrolü

In [ ]:
print(ev_df.isna().sum())


# 37. Korelasyon Tablosu

In [ ]:
print(
    ev_df.corr(numeric_only=True)
)


Korelasyon iki değişken arasındaki doğrusal ilişkinin yönü ve gücü hakkında fikir verebilir.

Ancak korelasyon tek başına neden-sonuç ilişkisi göstermez.


# 38. Alan ve Fiyat Grafiği

In [ ]:
plt.scatter(
    ev_df["Alan"],
    ev_df["Fiyat"]
)

plt.xlabel("Alan")
plt.ylabel("Fiyat")
plt.title("Alan ve Fiyat")
plt.show()


# 39. Yaş ve Fiyat Grafiği

In [ ]:
plt.scatter(
    ev_df["Yas"],
    ev_df["Fiyat"]
)

plt.xlabel("Bina Yaşı")
plt.ylabel("Fiyat")
plt.title("Bina Yaşı ve Fiyat")
plt.show()


# 40. Merkeze Uzaklık ve Fiyat

In [ ]:
plt.scatter(
    ev_df["MerkezeUzaklik"],
    ev_df["Fiyat"]
)

plt.xlabel("Merkeze Uzaklık")
plt.ylabel("Fiyat")
plt.title("Merkeze Uzaklık ve Fiyat")
plt.show()


# 41. Özellikleri ve Hedefi Ayırmak

In [ ]:
ozellikler = [
    "Alan",
    "Oda",
    "Yas",
    "MerkezeUzaklik"
]

X = ev_df[ozellikler]
y = ev_df["Fiyat"]

print(X.head())


# 42. Train-Test Ayrımı

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)


# 43. Çoklu Linear Regression Modeli

In [ ]:
ev_modeli = LinearRegression()

ev_modeli.fit(
    X_train,
    y_train
)

print("Model eğitildi.")


# 44. Katsayıları İncelemek

Birden fazla özellik olduğunda her özelliğin ayrı katsayısı bulunur.


In [ ]:
katsayilar = pd.DataFrame({
    "Ozellik": ozellikler,
    "Katsayi": ev_modeli.coef_
})

katsayilar


Katsayı işaretleri modelin doğrusal ilişkiyi hangi yönde öğrendiğini gösterir.

Ancak katsayıların nedensellik kanıtı olmadığını ve özelliklerin ölçeklerinden etkilendiğini unutmamalıyız.


# 45. Sabit Terim

In [ ]:
print(
    "Intercept:",
    ev_modeli.intercept_
)


# 46. Test Tahminleri

In [ ]:
ev_tahmin = ev_modeli.predict(
    X_test
)

print(ev_tahmin[:5])


# 47. Ev Modelinin Metrikleri

In [ ]:
ev_mae = mean_absolute_error(
    y_test,
    ev_tahmin
)

ev_mse = mean_squared_error(
    y_test,
    ev_tahmin
)

ev_rmse = np.sqrt(ev_mse)

ev_r2 = r2_score(
    y_test,
    ev_tahmin
)

print(f"MAE  : {ev_mae:,.2f}")
print(f"RMSE : {ev_rmse:,.2f}")
print(f"R²   : {ev_r2:.3f}")


# 48. Yeni Bir Ev İçin Tahmin

Örneğin:

- 140 m²,
- 3 oda,
- 7 yaş,
- merkeze 8 km

olan bir örnek için tahmin üretelim.


In [ ]:
yeni_ev = pd.DataFrame({
    "Alan": [140],
    "Oda": [3],
    "Yas": [7],
    "MerkezeUzaklik": [8.0]
})

yeni_fiyat = ev_modeli.predict(
    yeni_ev
)[0]

print(
    f"Tahmini fiyat: {yeni_fiyat:,.0f}"
)


Bu tahmin yalnızca derste oluşturduğumuz yapay veri kümesindeki örüntülere dayanır.

Gerçek emlak değerlemesi olarak kullanılmamalıdır.


# 49. Tahmin Fonksiyonu Yazmak

In [ ]:
def ev_fiyati_tahmin(
    alan,
    oda,
    yas,
    merkeze_uzaklik
):
    veri = pd.DataFrame({
        "Alan": [alan],
        "Oda": [oda],
        "Yas": [yas],
        "MerkezeUzaklik": [
            merkeze_uzaklik
        ]
    })

    tahmin = ev_modeli.predict(
        veri
    )[0]

    return tahmin


In [ ]:
sonuc = ev_fiyati_tahmin(
    180,
    4,
    5,
    4.5
)

print(
    f"Tahmini değer: {sonuc:,.0f}"
)


Bu fonksiyon daha sonra Tkinter veya Flask arayüzüne kolayca bağlanabilir.


# 50. Gerçek ve Tahmin Edilen Ev Fiyatları

In [ ]:
ev_sonuclari = X_test.copy()

ev_sonuclari["GercekFiyat"] = (
    y_test.to_numpy()
)

ev_sonuclari["TahminFiyat"] = (
    ev_tahmin
)

ev_sonuclari["Hata"] = (
    ev_sonuclari["GercekFiyat"]
    -
    ev_sonuclari["TahminFiyat"]
)

ev_sonuclari.head(10)


# 51. En Büyük Mutlak Hatalar

In [ ]:
ev_sonuclari["MutlakHata"] = (
    ev_sonuclari["Hata"].abs()
)

ev_sonuclari.sort_values(
    "MutlakHata",
    ascending=False
).head(10)


Modelin en çok hata yaptığı örnekleri incelemek, model geliştirme sürecinin önemli bir parçasıdır.


# 52. Residual Grafiği

In [ ]:
plt.scatter(
    ev_tahmin,
    y_test.to_numpy() - ev_tahmin
)

plt.axhline(
    y=0
)

plt.xlabel("Tahmin Edilen Fiyat")
plt.ylabel("Residual")
plt.title("Ev Modeli Residual Grafiği")
plt.grid()
plt.show()


# 53. Eğitim ve Test Skoru

Modelin eğitim ve test performanslarını birlikte incelemek faydalıdır.


In [ ]:
train_r2 = ev_modeli.score(
    X_train,
    y_train
)

test_r2 = ev_modeli.score(
    X_test,
    y_test
)

print("Train R²:", train_r2)
print("Test R² :", test_r2)


Eğitim başarısı çok yüksekken test başarısı belirgin biçimde düşükse overfitting ihtimali araştırılabilir.


# 54. Başka Bir Regresyon Modeli: Decision Tree

Doğrusal olmayan ilişkileri öğrenebilen bir model olarak Decision Tree Regressor deneyelim.


In [ ]:
from sklearn.tree import DecisionTreeRegressor

agac = DecisionTreeRegressor(
    max_depth=5,
    random_state=42
)

agac.fit(
    X_train,
    y_train
)

agac_pred = agac.predict(
    X_test
)

print(
    "Decision Tree R²:",
    r2_score(
        y_test,
        agac_pred
    )
)


# 55. Random Forest Regressor

Birden fazla karar ağacını birlikte kullanan Random Forest modelini de deneyebiliriz.


In [ ]:
from sklearn.ensemble import RandomForestRegressor

orman = RandomForestRegressor(
    n_estimators=100,
    random_state=42
)

orman.fit(
    X_train,
    y_train
)

orman_pred = orman.predict(
    X_test
)

print(
    "Random Forest R²:",
    r2_score(
        y_test,
        orman_pred
    )
)


# 56. Üç Modeli Karşılaştırmak

In [ ]:
model_sonuclari = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest"
    ],
    "MAE": [
        mean_absolute_error(
            y_test,
            ev_tahmin
        ),
        mean_absolute_error(
            y_test,
            agac_pred
        ),
        mean_absolute_error(
            y_test,
            orman_pred
        )
    ],
    "R2": [
        r2_score(
            y_test,
            ev_tahmin
        ),
        r2_score(
            y_test,
            agac_pred
        ),
        r2_score(
            y_test,
            orman_pred
        )
    ]
})

model_sonuclari


# 57. Model Karşılaştırma Grafiği

In [ ]:
plt.bar(
    model_sonuclari["Model"],
    model_sonuclari["R2"]
)

plt.ylabel("R²")
plt.title("Regresyon Modellerinin Karşılaştırılması")
plt.xticks(rotation=20)
plt.show()


En yüksek test skoruna sahip modeli otomatik olarak bulabiliriz.


In [ ]:
en_iyi = model_sonuclari.loc[
    model_sonuclari["R2"].idxmax()
]

print(en_iyi)


Tek bir train-test bölmesine göre en yüksek skoru bulmak faydalı bir ilk karşılaştırmadır.

Ancak daha güvenilir model seçimi için ilerleyen derste cross-validation kullanacağız.


# 58. KNN Regressor

KNN yalnızca sınıflandırma için kullanılmaz.

Sayısal hedef tahmini için `KNeighborsRegressor` kullanılabilir.


In [ ]:
from sklearn.neighbors import KNeighborsRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

knn_regresyon = Pipeline([
    (
        "scaler",
        StandardScaler()
    ),
    (
        "knn",
        KNeighborsRegressor(
            n_neighbors=7
        )
    )
])

knn_regresyon.fit(
    X_train,
    y_train
)

knn_pred = knn_regresyon.predict(
    X_test
)

print(
    "KNN R²:",
    r2_score(
        y_test,
        knn_pred
    )
)


KNN uzaklık kullandığı için özelliklerin ölçeklendirilmesi önemlidir.

Bu nedenle `StandardScaler` ve `KNeighborsRegressor` işlemlerini Pipeline içinde birleştirdik.


# 59. Tüm Modelleri Tek Tabloda Karşılaştırmak

In [ ]:
tum_modeller = pd.DataFrame({
    "Model": [
        "Linear Regression",
        "Decision Tree",
        "Random Forest",
        "KNN Regression"
    ],
    "MAE": [
        mean_absolute_error(
            y_test,
            ev_tahmin
        ),
        mean_absolute_error(
            y_test,
            agac_pred
        ),
        mean_absolute_error(
            y_test,
            orman_pred
        ),
        mean_absolute_error(
            y_test,
            knn_pred
        )
    ],
    "RMSE": [
        np.sqrt(mean_squared_error(
            y_test,
            ev_tahmin
        )),
        np.sqrt(mean_squared_error(
            y_test,
            agac_pred
        )),
        np.sqrt(mean_squared_error(
            y_test,
            orman_pred
        )),
        np.sqrt(mean_squared_error(
            y_test,
            knn_pred
        ))
    ],
    "R2": [
        r2_score(
            y_test,
            ev_tahmin
        ),
        r2_score(
            y_test,
            agac_pred
        ),
        r2_score(
            y_test,
            orman_pred
        ),
        r2_score(
            y_test,
            knn_pred
        )
    ]
})

tum_modeller.sort_values(
    "R2",
    ascending=False
)


# 60. Polynomial Regression Kavramına Giriş

Her ilişki düz bir çizgiyle açıklanamaz.

Örneğin:

```text
x arttıkça y önce yavaş, sonra hızlı artıyor
```

gibi eğrisel ilişkiler bulunabilir.

Polynomial Features kullanarak doğrusal regresyon modeline:

```text
x
x²
x³
```

gibi yeni özellikler ekleyebiliriz.


# 61. Eğrisel Veri Oluşturmak

In [ ]:
rng = np.random.default_rng(7)

x = np.linspace(
    -4,
    4,
    100
)

y = (
    3 * x ** 2
    + 2 * x
    + 8
    + rng.normal(
        0,
        4,
        len(x)
    )
)

polinom_df = pd.DataFrame({
    "x": x,
    "y": y
})

plt.scatter(
    polinom_df["x"],
    polinom_df["y"]
)

plt.title("Eğrisel Veri")
plt.show()


# 62. Düz Doğru ile Modellemek

In [ ]:
X_poly = polinom_df[["x"]]
y_poly = polinom_df["y"]

X_train_poly, X_test_poly, y_train_poly, y_test_poly = train_test_split(
    X_poly,
    y_poly,
    test_size=0.25,
    random_state=42
)

dogrusal = LinearRegression()

dogrusal.fit(
    X_train_poly,
    y_train_poly
)

dogrusal_pred = dogrusal.predict(
    X_test_poly
)

print(
    "Doğrusal R²:",
    r2_score(
        y_test_poly,
        dogrusal_pred
    )
)


# 63. Polynomial Features ile Model

In [ ]:
from sklearn.preprocessing import PolynomialFeatures

polinom_model = Pipeline([
    (
        "poly",
        PolynomialFeatures(
            degree=2,
            include_bias=False
        )
    ),
    (
        "linear",
        LinearRegression()
    )
])

polinom_model.fit(
    X_train_poly,
    y_train_poly
)

polinom_pred = polinom_model.predict(
    X_test_poly
)

print(
    "Polynomial R²:",
    r2_score(
        y_test_poly,
        polinom_pred
    )
)


# 64. Eğriyi Çizmek

In [ ]:
x_grafik = pd.DataFrame({
    "x": np.linspace(
        -4,
        4,
        200
    )
})

y_grafik = polinom_model.predict(
    x_grafik
)

plt.scatter(
    polinom_df["x"],
    polinom_df["y"],
    label="Veriler"
)

plt.plot(
    x_grafik["x"],
    y_grafik,
    label="Polynomial Model"
)

plt.legend()
plt.title("Polynomial Regression")
plt.show()


# 65. Model Karmaşıklığı

Polynomial derece arttıkça model daha karmaşık şekiller öğrenebilir.

Ancak gereğinden fazla karmaşıklık overfitting oluşturabilir.

Modelin amacı bütün eğitim noktalarından geçen bir eğri çizmek değil, yeni veriye genellenebilecek örüntüyü öğrenmektir.


# 66. Farklı Polynomial Dereceleri

Derece 1, 2 ve 8 modellerini karşılaştıralım.


In [ ]:
derece_sonuclari = []

for derece in [1, 2, 8]:
    m = Pipeline([
        (
            "poly",
            PolynomialFeatures(
                degree=derece,
                include_bias=False
            )
        ),
        (
            "linear",
            LinearRegression()
        )
    ])

    m.fit(
        X_train_poly,
        y_train_poly
    )

    train_skor = m.score(
        X_train_poly,
        y_train_poly
    )

    test_skor = m.score(
        X_test_poly,
        y_test_poly
    )

    derece_sonuclari.append({
        "Derece": derece,
        "Train_R2": train_skor,
        "Test_R2": test_skor
    })

pd.DataFrame(derece_sonuclari)


Train ve test sonuçlarını birlikte okumak model karmaşıklığını anlamada önemlidir.


# 67. Modeli Dosyaya Kaydetmek

Bir modeli kullanıcı her tahmin istediğinde yeniden eğitmek yerine eğitimden sonra dosyaya kaydedebiliriz.

Scikit-learn modelleri için `joblib` kullanılabilir.


In [ ]:
import joblib

joblib.dump(
    ev_modeli,
    "ev_fiyat_modeli.joblib"
)

print("Model kaydedildi.")


# 68. Modeli Tekrar Yüklemek

In [ ]:
yuklenen_model = joblib.load(
    "ev_fiyat_modeli.joblib"
)

print(yuklenen_model)


# 69. Yüklenen Model ile Tahmin

In [ ]:
test_ev = pd.DataFrame({
    "Alan": [125],
    "Oda": [3],
    "Yas": [10],
    "MerkezeUzaklik": [12]
})

sonuc = yuklenen_model.predict(
    test_ev
)[0]

print(
    f"Tahmin: {sonuc:,.0f}"
)


Bu yapı gerçek yapay zeka uygulamalarında çok önemlidir:

```text
Model Eğitimi
↓
Model Dosyası
↓
Tkinter / Flask / API
↓
Modeli Yükle
↓
Tahmin
```


# 70. Model Dosyalarında Güvenlik

`joblib` veya pickle tabanlı model dosyaları yalnızca güvenilen kaynaklardan yüklenmelidir.

Tanımadığınız veya güvenmediğiniz bir model dosyasını uygulamanıza yüklemek güvenli değildir.


# 71. Tahmin Raporu Fonksiyonu

In [ ]:
def tahmin_raporu(
    model,
    alan,
    oda,
    yas,
    uzaklik
):
    veri = pd.DataFrame({
        "Alan": [alan],
        "Oda": [oda],
        "Yas": [yas],
        "MerkezeUzaklik": [uzaklik]
    })

    tahmin = model.predict(
        veri
    )[0]

    return {
        "alan": alan,
        "oda": oda,
        "yas": yas,
        "merkeze_uzaklik": uzaklik,
        "tahmin": round(
            float(tahmin),
            2
        )
    }


In [ ]:
rapor = tahmin_raporu(
    yuklenen_model,
    160,
    4,
    6,
    5
)

print(rapor)


Bu sözlük daha sonra Flask API içinde JSON cevabına dönüştürülebilir.


# 72. Tkinter ile Regresyon Modeli

Masaüstü uygulamasında akış:

```text
Alan Entry
Oda Entry
Yaş Entry
Uzaklık Entry
↓
Tahmin Butonu
↓
Model.predict()
↓
Tahmini Fiyat Label
```

Önceki Tkinter bilgileri artık yapay zeka modeli için kullanıcı arayüzü oluşturmakta kullanılabilir.


# 73. Flask ile Regresyon Modeli

Web uygulamasında:

```text
HTML Form
↓
POST
↓
Flask Route
↓
joblib.load() ile model
↓
model.predict()
↓
Jinja
↓
Tahmin Sonucu
```

akışı kurulabilir.


# 74. API ile Regresyon Modeli

Bir API isteği:

```json
{
    "alan": 150,
    "oda": 3,
    "yas": 5,
    "uzaklik": 8
}
```

gönderebilir.

Model cevap olarak:

```json
{
    "tahmin": 4250000
}
```

benzeri bir sonuç döndürebilir.

İlerleyen derslerde model servisleri ve API tasarımına geçeceğiz.


# 75. Regresyon Modelinde Veri Kalitesi

Gerçek bir model geliştirirken şu sorular sorulmalıdır:

- Veride eksik kayıt var mı?
- Özelliklerin birimleri doğru mu?
- Hatalı değerler var mı?
- Aykırı değerler var mı?
- Eğitim verisi gerçek kullanım alanını temsil ediyor mu?
- Hedef değer doğru ölçülmüş mü?
- Veri güncel mi?

İyi algoritma kötü veriyi otomatik olarak iyi hale getirmez.


# 76. Tahmin Aralığının Dışına Çıkmak

Model eğitim verisinde:

```text
60 - 250 m²
```

evler gördüyse:

```text
2000 m²
```

bir değer için tahmin üretmesi teknik olarak mümkün olabilir.

Ancak bu tahmin güvenilir olmayabilir.

Bu nedenle gerçek uygulamalarda giriş değerlerinin eğitim verisinin mantıklı sınırları içinde olup olmadığı kontrol edilmelidir.


# 77. Tahmin ve Karar Arasındaki Fark

Model çıktısı bir **tahmindir**.

Örneğin:

```text
Tahmini değer = 4.250.000
```

Bu değer:

- kesin gerçek,
- garanti,
- resmi değerleme

değildir.

Yapay zeka uygulaması tasarlarken kullanıcıya model çıktısının ne anlama geldiğini doğru anlatmak önemlidir.


# 78. Regresyon Uygulamalarında Hedef Sızıntısı

Modelin tahmin edeceği sonucu doğrudan veya dolaylı olarak içeren bir özelliği modele vermek değerlendirmeyi yanıltabilir.

Örneğin fiyat tahmini yaparken:

```text
NihaiSatışFiyatı
```

gibi hedefi zaten açığa çıkaran bir sütunu özellik olarak kullanmak veri sızıntısıdır.

Özelliklerin tahmin anında gerçekten erişilebilir olması gerekir.


# 79. Model Seçiminde Tek Test Setinin Sınırı

Bu derste modelleri tek train-test bölmesiyle karşılaştırdık.

Ancak farklı rastgele bölmeler farklı sonuçlar üretebilir.

Bir sonraki model değerlendirme dersinde:

- cross-validation,
- validation,
- GridSearchCV,
- hiperparametre optimizasyonu

konularıyla daha sağlam karşılaştırma yapacağız.


# 80. Regresyon Projesinin Genel Akışı

```text
Problem Tanımla
↓
Sayısal Hedef Belirle
↓
Veriyi Topla
↓
Veriyi İncele
↓
X ve y Oluştur
↓
Train / Test Ayır
↓
Baseline Oluştur
↓
Model Eğit
↓
predict()
↓
MAE / RMSE / R²
↓
Hata Analizi
↓
Model Karşılaştır
↓
Yeni Veri Tahmini
↓
Modeli Kaydet
↓
Uygulamaya Entegre Et
```


# 81. Ders Özeti

Bu derste:

- regresyon,
- sınıflandırma-regresyon farkı,
- doğrusal regresyon,
- `LinearRegression`,
- `coef_`,
- `intercept_`,
- `fit()`,
- `predict()`,
- residual,
- train-test ayrımı,
- MAE,
- MSE,
- RMSE,
- R²,
- baseline model,
- `DummyRegressor`,
- çoklu doğrusal regresyon,
- özellik katsayıları,
- hata analizi,
- Decision Tree Regressor,
- Random Forest Regressor,
- KNN Regressor,
- StandardScaler,
- Pipeline,
- model karşılaştırma,
- Polynomial Regression,
- model karmaşıklığı,
- overfitting,
- `joblib`,
- model kaydetme,
- model yükleme,
- uygulamaya model entegrasyonu

konularını öğrendik.


# 82. Mini Uygulamalar

1. Çalışma süresi ve puan verisi oluşturun.
2. Veriyi scatter grafikle gösterin.
3. Linear Regression modeli eğitin.
4. Modelin katsayısını yazdırın.
5. Sabit terimi yazdırın.
6. 6.5 saat çalışma için puan tahmini yapın.
7. Train-test ayrımı uygulayın.
8. MAE hesaplayın.
9. MSE hesaplayın.
10. RMSE hesaplayın.
11. R² hesaplayın.
12. Gerçek ve tahmin değerlerini DataFrame'de karşılaştırın.
13. Residual sütunu oluşturun.
14. Residual grafiği çizin.
15. DummyRegressor ile baseline oluşturun.
16. Baseline ile Linear Regression modelini karşılaştırın.
17. En az üç özellikli yapay fiyat veri kümesi oluşturun.
18. Çoklu Linear Regression eğitin.
19. Katsayıları DataFrame halinde gösterin.
20. Decision Tree Regressor deneyin.
21. Random Forest Regressor deneyin.
22. KNN Regressor için Pipeline oluşturun.
23. En az dört modeli MAE ve R² ile karşılaştırın.
24. Polynomial Regression uygulaması geliştirin.
25. Eğitilmiş modeli `joblib` ile kaydedip yeniden yükleyin.


# 83. Yapay Zeka Proje Görevi

Bir **Sayısal Tahmin Sistemi** geliştirin.

Konu seçenekleri:

- öğrenci puanı tahmini,
- yapay ev fiyatı tahmini,
- ürün talep tahmini,
- enerji tüketimi tahmini,
- günlük satış tahmini,
- sensör ölçümü tahmini.

Projede en az:

- 200 örnek,
- 3 özellik,
- 1 sayısal hedef,
- Pandas veri analizi,
- en az 2 grafik,
- train-test ayrımı,
- baseline model,
- Linear Regression,
- en az iki farklı regresyon modeli,
- MAE,
- RMSE,
- R²,
- model karşılaştırma tablosu,
- hata analizi,
- yeni veri tahmini,
- model dosyasına kaydetme

bulunsun.

Ek geliştirme:

- Tkinter tahmin ekranı,
- Flask tahmin formu,
- JSON API,
- tahmin geçmişini SQLite'a kaydetme

özelliklerinden biri eklenebilir.


# Dersin Ana Kazanımı

Bu dersin sonunda öğrencinin aşağıdaki regresyon zincirini kurabilmesi hedeflenmektedir:

**Sayısal Tahmin Problemi**

↓

**Özellikler ve Hedef**

↓

**Veri Analizi**

↓

**Train / Test**

↓

**Baseline**

↓

**Regresyon Modeli**

↓

**fit()**

↓

**predict()**

↓

**MAE / RMSE / R²**

↓

**Hata Analizi**

↓

**Model Karşılaştırması**

↓

**Yeni Veri Tahmini**

↓

**Modeli Kaydetme**

↓

**Masaüstü / Web / API Entegrasyonu**

Artık yapay zeka uygulamalarımız yalnızca sınıf tahmini değil, gerçek sayısal değer tahminleri de üretebilmektedir.

Bir sonraki derste farklı **sınıflandırma algoritmalarını** daha ayrıntılı inceleyip modelleri sistematik biçimde karşılaştıracağız.
